<h1 style="text-align: center;">HEISENBERG-BASED FAULT LOCALIZATION (HBFL)</h1>

<h2 style="text-align: center;">Download Initial Dependencies</h2>

In [1]:
import numpy as np
import pandas as pd
import json
import math
from tqdm import tqdm
from qiskit.quantum_info import SparsePauliOp, Operator, Pauli, Clifford
import qiskit.qasm2 as q2
from qiskit import QuantumCircuit
from IPython.display import display

from heisenbergUtilities import *

In [2]:
# from qiskit_aer import AerSimulator
# simulator = AerSimulator()

# print(simulator.available_devices())

<h2 style="text-align: center;">CONTROL PANEL</h2>

In [ ]:
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#-------------------------------------------------------------------------GENERAL PARAMETERS---------------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
ALG_NAME = "qaoa"
ORIGINAL_QASM = f"{ALG_NAME}_5_qubits.qasm"                                                 #| The original algorithm QASM circuit file name.                           
ORIGINAL_QASM_FOLDER = "SBFL_circuits/MQTBench/"                                            #| The folder containing the original QASM circuit file.                    
QASM_MUTANT_FOLDER = f"SBFL_circuits/QMutBenchMutants/Mutants_{ALG_NAME}_5_qubits/"         #| The folder containing all mutants related to the original QASM circuit.  
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#-------------------------------------------------------------------------SB-QOPS PARAMETERS---------------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
RUNS = 10                                                                                   #| Number of times SB-QOPS will run to find a failing or passing test case.            
SHOTS = 20000                                                                               #| Number of shots for SB-QOPS to use to create the compact program specification.     
EPOCH = 80                                                                                  #| Number of epochs SB-QOPS will run to navigate the search space of test cases.       
SBQOPS_TOL = 0.1                                                                            #| Tolerance value SB-QOPS uses to determine if a test case is failing or passing.     
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#---------------------------------------------------------------------HEISENBERG SBFL PARAMETERS-----------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
LAMBDA_PHASE = 2                                                                        #| The hyperparameter used to weight the phase change of the Pauli propagation
LAMBDA_CHANGE = 1                                                                         #| The hyperparameter used to weight the change of string of the Pauli propagation
LAMBDA_SIMILARITY = 1                                                                       #| The hyperparameter used to weight the similarity difference between current and target Pauli
ATOL = 1e-4                                                                                 #| The tolerance value used to prune negligible magnitudes during Pauli propagation.   
MAX_TERMS = 100                                                                             #| The maximum number of terms to keep during Pauli propagation. If None, use all.     
SEARCH_STEP = 3                                                                             #| The search step size used during Pauli propagation. If None, pauli-prop uses 4.     
VERBOSE = 1                                                                                 #| A boolean value to determine if the program should print out detailed information.  
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#


Generate the linked list of all operations that take place in the inverse circuit

In [4]:
"""
LinkedListNode: A class that holds information related to a gate in a quantum circuit. 

PROPERTIES:
    - value (String): The name of the quantum gate
    - depth (int): The depth of the gate in the circuit
    - count (Int): The probability that the gate will meaningfully change the Pauli string.
    - next (LinkedListNode): The next gate in the circuit

"""

class LinkedListNode:
    def __init__(self, name="Initial", depth=0, count=0,  next=None):
        self.value = name
        self.depth = depth
        self.next = next
        self.count = count

class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, name, depth):
        new_node = LinkedListNode(name, depth)
        if not self.head:
            self.head = new_node
            return
        last_node = self.head
        while last_node.next:
            last_node = last_node.next
        last_node.next = new_node

    def mark(self, depth):
        current_node = self.head
        while current_node:
            if current_node.depth == depth:
                current_node.count += 1
                return
            current_node = current_node.next

    def reset(self):
        current_node = self.head
        while current_node:
            current_node.count = 0
            current_node = current_node.next

<h3>Download MQT Benchmark circuits.</h3>

We'll start with DJ, RealAmpRandom, and GHZ since SB-QOPS caught those mutants 100% of the time for 5 qubit circuits



<h3>Generate mutants.</h3>

We'll start with mutants that add an unnecessary gate, then add mutants that replace a certain gate later.

The mutants were downloaded from QMutBench, choosing specifically mutants that added a gate anywhere with 0-10% survival scores

In [5]:
correct_circuit = load_program(ORIGINAL_QASM, ORIGINAL_QASM_FOLDER).copy()
correct_list = LinkedList()
correct_list = construct_list(correct_list, correct_circuit, inverse=False)

<h2 style="text-align: center;"> SB-QOPS </h2>

This is where we will use SB-QOPS on a provided circuit and its mutants

For a single mutant, it took about 2 minutes to generate a 20 test suite (10 fail, 10 pass) on an RTX 4090 SUPER. It can only be run on a Linux environment since the AER Simulator does not support Windows

There are 113 mutants for the DJ algorithm in use: 226 minutes or 3 3/4 hours

There are 89 mutants for the GHZ algorithm in use: 178 minutes or about 3 hours

There are 125 mutants for the RealAmpRandom algorithm in use: 250 minutes or just over 4 hours

BUT this cell should only need to be run once per algorithm with mutants under test. After it saves to the csv file at the end, this cell can be commented out as to not accidentally run it again. 

In the future if we want to test this methodology on higher qubit circuits or other algorithms, we'll likely either want to reduce the number of tests in the suite or the number of mutants we're analyzing

In [6]:
# import QOPS as qops
# from QOPS_test import get_compact_program_specification_Z
# from pathlib import Path

# ga_result = pd.DataFrame(columns=['Program',"path_to_mutant",'catch_avg','avg_fitness','failing_testcases', 'passing_testcases'])
# program_history = {}

# #program_specification = The compact program specification. SB-QOPS needs this to generate failing test cases.
# # In a public-use environment, this would be provided by the user.
# program_specification = get_compact_program_specification_Z(correct_circuit, shots=SHOTS)

# #mutants = A python list of qiskit QuantumCircuit variables with a mutation in them
# mutants = []
# mutants_names = []
# root = Path(QASM_MUTANT_FOLDER)
# for qasm_mutant in root.rglob("*.qasm"):
#     mutants.append(load_program(qasm_mutant.name, QASM_MUTANT_FOLDER))
#     mutants_names.append(qasm_mutant.name)

# for mutant_index,mutant in enumerate(mutants):
#     tester = qops.Circuit_Tester(CUT=mutant)
#     tester.set_applicable_families_Z(program_specification)
#     mutants_per_run = []
#     fitness_per_run = []
#     failing_testcases_per_run = []
#     history_per_run = []

#     print("NOW RUNNING TO FIND FAILING TESTS")
#     #For a predefined number of attempts, try to find a set of failing test cases (output is above predefined tolerance)
#     for runs in range(RUNS):
#         killed = 0
#         pauli = {}
#         fitness = 0
#         for i in range(len(tester.applicable_families)):
#             best_function,testcase, history = tester.run_mealoneplusone(i, EPOCH)
#             if best_function > SBQOPS_TOL:
#                 killed = 1
#                 pauli = testcase
#                 fitness = best_function
#                 break
#         mutants_per_run.append(killed)
#         failing_testcases_per_run.append(pauli)
#         fitness_per_run.append(fitness)
#         history_per_run.append(history)

#     avg_score = np.mean(mutants_per_run)
#     avg_fitness = np.mean(fitness_per_run)

#     #Here is our new, modified algorithm from the SB-QOPS method that attempts to find passing test cases as well. We'll need them for SBFL
#     passing_testcases_per_run = []

#     print("NOW RUNNING TO FIND PASSING TESTS")
#     for runs in range(RUNS):
#         pauli = {}
#         fitness = 0
#         for i in range(len(tester.applicable_families)):
#             best_function, testcase, history = tester.run_reverse_mealoneplusone(i,EPOCH)
#             if best_function < SBQOPS_TOL:
#                 pauli = testcase
#                 break
#         passing_testcases_per_run.append(pauli)

#     #Replace static program file with "program_name" when ready to test fully
#     """
#     ga_result: A pandas dataframe with the following columns
#         Program: Name of the mutant quantum circuit file
#         Path_To_Mutant: Path to the mutant file
#         Catch_Avg: The average number of times the mutant was identified by SB-QOPS
#         Avg_Fitness: The average fitness of the search algorithm used. Higher usually indicates higher Catch_Avg
#         Failing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a failing test case
#         Passing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a passing test case
#     """
#     ga_result = pd.concat([pd.DataFrame([[mutants_names[mutant_index],QASM_MUTANT_FOLDER,avg_score,avg_fitness,json.dumps(failing_testcases_per_run), json.dumps(passing_testcases_per_run)]],columns=ga_result.columns),ga_result],ignore_index=True)
#     ga_result.to_csv(f'realamprandom_ga_result', index=False)

ga_result is a slightly altered pandas Dataframe with similar structure found in SB-QOPS's results folder. The differences are changing the column "Program" to be the name of the mutant file instead of the original quantum circuit, changing "Mutant" to be the path to the mutant file instead of being an arbitrary index value, and adding a passing_testcases column that returns Pauli strings and coefficients that generate passing tests.

Now we want to run SBFL using Heisenberg evolution trees on each circuit placed in ga_result

Evolution analysis tends to take about 5 minutes for 10 failing tests on a very complex algorithm such as QNN, so about 10 minutes for 20 total tests

DJ was a relatively small algorithm with few to no branching gates, so it only ended up taking about 5-6 minutes to evolve and analyze all 113 DJ mutants

About 890 minutes for GHZ, or 14.8 hours to evolve and analyze all 89 GHZ mutants

About 1250 minutes for RealAmpRandom, or 20.8 hours to evolve and analyze all 125 RealAmpRandom mutants

The runtime of this code segment is directly tied to the depth of the circuit being analyzed and the number of 2-branching gates such as rotation or phase gates that exist in the circuit.

This is something to note in the final paper. Regardless of accuracy this methodology takes a long time to run. If results are promising, then we'll want to look into the tradeoffs between EXAM and hyperparameters to speed this up. Primarily the atol, max_terms, and search_step parameters in the evolve_pauli_exact method in HeisenbergUtilities.py


<h2 style="text-align: center;">HEISENBERG EVOLUTIONS (PAULI PROPAGATION)</h2>

In [7]:
from heisenbergTree import *

ga_result = pd.read_csv(f'{ALG_NAME}_ga_result.csv')
tarantula_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])
barinel_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])
dstar_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])
mutant_num = 0

#For each mutant of a circuit, run the SBFL implementation
for mutant, path in zip(ga_result.loc[:,"Program"], ga_result.loc[:,"path_to_mutant"]):
    print("Now evolving the following mutant: ", mutant)

    #Extract the raw test cases found for each mutant
    desired_failing_testcases = ga_result.loc[(ga_result["Program"] == mutant), ["failing_testcases"]]
    desired_passing_testcases = ga_result.loc[(ga_result["Program"] == mutant), ["passing_testcases"]]

    dict_failing_testcases = desired_failing_testcases.to_dict()
    dict_passing_testcases = desired_passing_testcases.to_dict()

    string_failing_testcases = dict_failing_testcases['failing_testcases'][mutant_num]
    string_passing_testcases = dict_passing_testcases['passing_testcases'][mutant_num]

    list_failing_testcases = json.loads(string_failing_testcases)
    list_passing_testcases = json.loads(string_passing_testcases)

    #Remove empty tests
    raw_failing_testcases = remove_null_tests(list_failing_testcases)
    raw_passing_testcases = remove_null_tests(list_passing_testcases)

    circuit_inverse = load_program(mutant,path).copy().inverse()  # Invert to track backward evolution of the operator
    forward_mutant = load_program(mutant, path).copy()

    #Create the linked list of operations in the inverse circuit
    operation_list = LinkedList()
    operation_list = construct_list(operation_list, circuit_inverse, True)

    forward_list = LinkedList()
    forward_list = construct_list(forward_list, forward_mutant, False)

    #Create list of nodes in linked list. Somewhere down the road remove the linked list implementation. It's unnecessary and giving me a headache
    node_list = []
    node = operation_list.head
    while node:
        node_list.append(node)
        node = node.next

    #Perform Pauli propagation (Heisenberg evolution) for all test cases. Returns a dataframe with all counts for all operations
    global_frame_fail = heisenberg_evolve(circuit_inverse, 
                                          operation_list, 
                                          raw_failing_testcases, 
                                          "fail", 
                                          LAMBDA_PHASE, 
                                          LAMBDA_CHANGE, 
                                          LAMBDA_SIMILARITY, 
                                          atol = ATOL, 
                                          max_terms = MAX_TERMS, 
                                          search_step = SEARCH_STEP)
    
    global_frame_pass = heisenberg_evolve(circuit_inverse, 
                                          operation_list, 
                                          raw_passing_testcases, 
                                          "pass", 
                                          LAMBDA_PHASE, 
                                          LAMBDA_CHANGE, 
                                          LAMBDA_SIMILARITY, 
                                          atol = ATOL, 
                                          max_terms = MAX_TERMS, 
                                          search_step = SEARCH_STEP)

    global_frame = pd.concat([global_frame_fail, global_frame_pass], ignore_index=True)

    global_frame = global_frame.drop(["Initial"],axis=1)
    if VERBOSE:
        display(global_frame)

    tarantula_scores = tarantula(global_frame)
    barinel_scores = barinel(global_frame)
    dstar_scores = dstar(global_frame)

    if VERBOSE:
        print("BARINEL SCORES")
        display(barinel_scores)
        print("TARANTULA SCORES")
        display(tarantula_scores)
        print("DSTAR SCORES")
        display(dstar_scores)
    
    # Here is where analysis of the methodology begins. 
    # We first extract the position of the erroneous gate by comparing it to the original MQT gate
    # NOTE: This method is intended for single-added gates for now. Other types of mutants will require other methods later
    error_gate_location = find_erroneous_gate(forward_mutant, correct_circuit)

    if VERBOSE:
        print("ERROR GATE LOCATION:")
        print(error_gate_location)

    # Calculate the EXAM score for Barinel by counting the number of gates we would have to observe before we reach the erroneous gate.
    gates_observed = 0
    barinel_exam_score = 0
    for col_name, col_date in barinel_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            barinel_exam_score = gates_observed/(circuit_inverse.size())
            break

    # Calculate the EXAM score for Tarantula by counting the number of gates we would have to observe before we reach the erroneous gate.
    gates_observed = 0
    tarantula_exam_score = 0
    for col_name, col_date in tarantula_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            tarantula_exam_score = gates_observed/(circuit_inverse.size())
            break

    # Calculate the EXAM score for DStar by counting the number of gates we would have to observe before we reach the erroneous gate.
    gates_observed = 0
    dstar_exam_score = 0
    for col_name, col_date in dstar_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            dstar_exam_score = gates_observed/(circuit_inverse.size())
            break

    # Here we store the results from the HBFL analysis for both barinel and tarantula into a csv file for later analysis.
    barinel_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,barinel_exam_score]],columns=barinel_sbfl_result.columns),barinel_sbfl_result],ignore_index=True)
    barinel_sbfl_result.sort_values(by=['exam_score'], ascending=False)
    barinel_sbfl_result.to_csv(f'{ALG_NAME}_barinel_sbfl_result.csv', index=False)

    tarantula_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,tarantula_exam_score]],columns=tarantula_sbfl_result.columns),tarantula_sbfl_result],ignore_index=True)
    tarantula_sbfl_result.sort_values(by=['exam_score'], ascending=False)
    tarantula_sbfl_result.to_csv(f'{ALG_NAME}_tarantula_sbfl_result.csv', index=False)

    dstar_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,dstar_exam_score]],columns=dstar_sbfl_result.columns),dstar_sbfl_result],ignore_index=True)
    dstar_sbfl_result.sort_values(by=['exam_score'], ascending=False)
    dstar_sbfl_result.to_csv(f'{ALG_NAME}_dstar_sbfl_result.csv', index=False)

    mutant_num += 1

if VERBOSE:
    display(barinel_sbfl_result)
    display(tarantula_sbfl_result)
    

Now evolving the following mutant:  AddGate_y_inGap_23_.qasm


100%|██████████| 10/10 [00:18<00:00,  1.86s/it]


,ry 23,y 22,ry 21,ry 20,ry 19,ry 18,cx 17,cx 16,cx 15,cx 14,...,cx 8,cx 7,cx 6,cx 5,ry 4,ry 3,ry 2,ry 1,ry 0,pass/fail
0,0.672966,0.476569,1.179591,0.893927,2.291129,1.885169,2.027543,1.976690,1.961802,1.999661,...,0.288643,0.674824,0.769131,0.546005,0.240277,0.019246,0.023823,0.261490,0.000007,fail
1,0.633580,0.448677,1.235501,0.712118,1.792391,1.955925,2.091412,1.716144,1.657443,1.686632,...,0.224182,0.550116,0.591625,0.433785,0.216714,0.015403,0.019184,0.194335,0.000006,fail
2,0.622212,0.440627,1.279954,0.549914,1.315406,2.109780,2.340276,2.100459,1.955884,1.683323,...,0.205982,0.600449,0.603157,0.402844,0.180223,0.018436,0.019717,0.208038,0.000006,fail
3,0.690226,0.488792,1.385134,0.851970,2.061784,2.436842,2.618374,2.211791,2.355052,2.312164,...,0.271375,0.746097,0.799958,0.617708,0.307666,0.019078,0.025675,0.262429,0.000008,fail
4,0.372694,0.263928,0.861154,0.618990,1.514276,1.839159,2.081991,2.048568,2.218300,2.278984,...,0.306430,0.693569,0.786102,0.573294,0.260896,0.018363,0.025805,0.256333,0.000007,fail
5,0.617312,0.437157,1.232941,0.754792,1.514766,2.450855,2.769213,2.450721,2.373098,2.123554,...,0.291539,0.707415,0.740858,0.575458,0.268978,0.019398,0.024589,0.257404,0.000009,fail
6,0.557218,0.394601,1.366182,0.612053,1.871638,2.580139,2.707979,2.187734,1.924200,1.750619,...,0.235623,0.560886,0.561892,0.495808,0.252912,0.018570,0.019006,0.202907,0.000007,fail
7,0.669824,0.474344,1.318906,0.773198,2.045405,1.609124,1.708871,1.883707,1.583354,1.693860,...,0.284974,0.611845,0.640270,0.485129,0.246327,0.017508,0.020069,0.232883,0.000005,fail
8,0.543127,0.384622,1.083937,0.743447,1.920294,1.754790,1.836425,1.776362,1.727586,1.543027,...,0.273029,0.517793,0.589350,0.429254,0.198672,0.017457,0.019417,0.187018,0.000005,fail
9,0.634500,0.449329,1.300844,0.565605,1.522333,1.832290,1.951940,1.818234,1.813629,1.386257,...,0.149756,0.428768,0.445146,0.360081,0.189604,0.014627,0.015628,0.138636,0.000005,fail


BARINEL SCORES


,ry 23,y 22,ry 3,ry 12,ry 10,ry 9,cx 16,cx 15,ry 21,cx 17,...,cx 14,ry 0,ry 1,cx 7,cx 8,ry 19,cx 5,ry 4,ry 13,ry 20
sum,0.527838,0.527838,0.526534,0.521743,0.52039,0.519451,0.516441,0.515222,0.50908,0.508025,...,0.503265,0.502202,0.501021,0.498578,0.493502,0.488834,0.486283,0.473027,0.468913,0.461018


...

DSTAR SCORES


,cx 15,cx 17,cx 16,ry 20,ry 18,cx 14,ry 22,ry 21,ry 12,cx 6,...,z 19,ry 13,ry 4,cx 8,ry 1,ry 2,ry 3,ry 0,ry 10,ry 11
sum,15.625883,15.184403,14.907013,14.235544,13.887808,12.604379,5.735489,2.734601,2.719971,2.578067,...,1.530372,1.370611,0.488525,0.46466,0.405858,0.004415,0.003087,3.920834e-10,1.160490e-10,2.579398e-11


ERROR GATE LOCATION:
19


,Program,path_to_mutant,exam_score
0,AddGate_z_inGap_20_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.041667
1,AddGate_ry_inGap_17_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.083333
2,AddGate_y_inGap_26_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.625
3,AddGate_y_inGap_20_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.041667
4,AddGate_ry_inGap_16_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.291667
...,...,...,...
86,AddGate_x_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.166667
87,AddGate_y_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.125
88,AddGate_y_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.166667
89,AddGate_ry_inGap_15_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.291667


,Program,path_to_mutant,exam_score
0,AddGate_z_inGap_20_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.041667
1,AddGate_ry_inGap_17_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.083333
2,AddGate_y_inGap_26_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.625
3,AddGate_y_inGap_20_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.041667
4,AddGate_ry_inGap_16_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.291667
...,...,...,...
86,AddGate_x_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.166667
87,AddGate_y_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.125
88,AddGate_y_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.166667
89,AddGate_ry_inGap_15_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_vqe_5_q...,0.291667
